# (Des)carga de base de datos

## Instalación y carga de la librería del repositorio UCI ML

Instalar la librería del repositorio utilizando pip.

In [ ]:
pip install ucimlrepo

Verificar que la librería del repositorio se encuentra correctamente instalada.

In [ ]:
pip list | grep uci

Cargar la librería en la sesión actual de python.

In [ ]:
from ucimlrepo import fetch_ucirepo

Cargar librerías auxiliares

In [ ]:
# Importar clase Path para utilizar rutas en directorios
from pathlib import Path
# Importar pandas para cargar base como dataframe
import pandas as pd
# Importar numpy para métodos de agregación
import numpy as np

## Carga de la base de datos de cirrosis

Según la documentación del repositorio, nuestra base de datos de interés (cirrosis) se encuentra indexada con el id 878.

In [ ]:
cirrhosis_patient_survival_prediction = fetch_ucirepo(id=878)

Ver cómo viene la información

In [ ]:
cirrhosis_patient_survival_prediction

Estructurar la información en un data frame

In [ ]:
X = cirrhosis_patient_survival_prediction.data.features
y = cirrhosis_patient_survival_prediction.data.targets
cirrosis = pd.concat([X,y], axis=1)

Ver las primeras observaciones

In [ ]:
cirrosis.head(10)

## En caso que fracasemos con la librería uci

Ruta de trabajo actual para movernos entre directorios y poder leer la base correctamente.

In [ ]:
BASE_DIR = Path().resolve().parent

Ver cual es el directorio al que apunta BASE_DIR

In [ ]:
BASE_DIR

Ruta de del directorio donde se encuentra el archivo de cirrosis 

In [ ]:
DATA_DIR = BASE_DIR / 'data' / 'cirrosis'
DATA_DIR

Leer el archivo csv que se encuentra en el directorio de cirrosis utilizando pandas para estructurarlo en un data frame.

In [ ]:
cirrosis2 = pd.read_csv(DATA_DIR / 'cirrhosis.csv')

Ver las primeros renglones del data frame.

In [ ]:
cirrosis2.head(10)

## Sanity check

Verificar que tienen la misma dimensión

In [ ]:
cirrosis.shape

In [ ]:
cirrosis2.shape

Observamos que en cirrosis2 existen un par de columnas que no se encontraban en cirrosis:

In [ ]:
set(cirrosis2.columns).symmetric_difference(set(cirrosis.columns))

La columna 'ID' sirve de identificador por lo cual no representa ninguna diferencia significativa que pudiera afectar el análisis de nuestros datos. ¿Pero que pasa con 'N_Days'?

**Revisando la documentación** encontramos que dicha variable representaba el tiempo transcurrido desde que se registraban al estudio a que pasaba uno de los siguientes escenarios:
1. Muerte (D)
2. Trasplante (CL)
3. Fin del estudio (C)

Cómo nosotros no estamos haciendo el análisis de supervivencia y tomamos los datos con fines meramente de estudio, vamos a considerar N_Days también como variable explicativa.

## "Arreglo" de las bases

Primero voy a buscar como agregar la columna N_Days a mi base cirrosis. Explorando el objeto cirrhosis_patient_survival_prediction vemos que en data había un original que es idéntica a la tabla generada con cirrosis2.

In [ ]:
cirrosis = cirrhosis_patient_survival_prediction.data.original
cirrosis.head(10)

In [ ]:
all(cirrosis == cirrosis2)

De esta manera, podemos quedarnos solo con cirrosis, "descartando" la columna 'ID', y eliminar el objeto cirrosis2.

OJO: Dependiendo de la base que pudieron descargar, a esa le "descartan" la columna 'ID' y la otra no existía entonces no la borran.

In [ ]:
cirrosis.set_index('ID', inplace=True)
cirrosis.head(2)

In [ ]:
del cirrosis2

Reordenaremos las columnas para tener la variable de interés (Status) al final.

In [ ]:
cirrosis = cirrosis[[x for x in cirrosis.columns if x != 'Status'] + ['Status']]
cirrosis.head()

# Filtrado, ordenamiento, mezclas y agregación

## Filtrado de filas de acuerdo a condiciones lógicas

En ocasiones nos interesan ciertas observaciones por lo que necesitamos sera capaces de extraerlas. Veamos las distintas etapas que existen para luego filtrar las observaciones correspondientes con la etapa inicial de la enfermedad.

In [ ]:
cirrosis.Stage.value_counts()

In [ ]:
cirrosis_etapa_1 = cirrosis[cirrosis['Stage'] < 2]
cirrosis_etapa_1.head(5)

In [ ]:
cirrosis_etapa_1.Stage.value_counts()

OJO: Cuidado con el tipo de dato que tienen porque en ocasiones al no tomarlo en consideración las operaciones y los filtros pueden no comportarse de manera adecuada.

In [ ]:
cirrosis[cirrosis['Stage'] == '1.0']

En ocasiones se pueden necesitar filtros que dependan de más de una variables; por ejemplo, menores de edad en etapa 1 de género F. Hay que tener cuidado pues la edad esta en días.

In [ ]:
stage_mask = (cirrosis['Stage'] < 2)
stage_mask

Para verificar que funciona bien la máscara, pueden sumar los boleanos y deberían ser 21

In [ ]:
stage_mask.sum()

In [ ]:
age_mask = (cirrosis['Age'] / 365 < 18)
sex_mask = (cirrosis['Sex'] == 'F')

Filtro las observaciones que cumplan con todos los requisitos.

In [ ]:
cirrosis_F_menores_edad = cirrosis[stage_mask & age_mask & sex_mask]

Equivalentemente podrían escribirlo como sigue

In [ ]:
cirrosis[(cirrosis['Stage'] < 2) & (cirrosis['Age'] / 365 < 18) & (cirrosis['Sex'] == 'F')]

In [ ]:
cirrosis_F_menores_edad

Observamos que en el estudio no hay menores de edad

In [ ]:
(cirrosis['Age'] / 365 < 18).sum()

Nos enteramos que no es legal hacer estudios clínicos experimentales en niños (sin consentimiento de sus tutores legales y en casos no extremos) por lo cual nuestra base no tiene información de menores de edad.

Bueno... vamos a relajar un poco el supuesto y digamos que queremos menores de edad o mayores de 65 años.

In [ ]:
age_mask_2 = age_mask | (cirrosis['Age'] / 365 > 65)
cirrosis_F_no_laboral = cirrosis[stage_mask & sex_mask & age_mask_2]

Equivalentemente...

In [ ]:
cirrosis[(cirrosis['Stage'] < 2) & ((cirrosis['Age'] / 365 < 18) | (cirrosis['Age'] / 365 > 65)) & (cirrosis['Sex'] == 'F')]

In [ ]:
cirrosis_F_no_laboral

Haciendo un análisis de la información vemos que el rango de edad en pacientes género 'F' que se encuentran en etapa 1 esta entre 28 y 62 años. Por lo cual hace sentido que no recuperemos información.

In [ ]:
(cirrosis[stage_mask].Age /365).describe()

In [ ]:
(cirrosis[sex_mask].Age /365).describe()

In [ ]:
(cirrosis[age_mask_2].Age /365).describe()

## Selección de variables (columnas)

In [ ]:
cirrosis.head(2)

In [ ]:
cirrosis.columns

Guardar nombres de distintas variables en listas dependiendo de su tipo o significado.

In [ ]:
signos_clinicos = ['Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
nivels_sangre = ['Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
variables_categoricas_nominales = ['Drug', 'Sex', 'Status']
variables_categoricas_ordinales = ['Stage']
variables_continuas = ['N_Days', 'Age']

Sanity check que no elimine variables...

In [ ]:
len(signos_clinicos) + len(nivels_sangre) + len(variables_categoricas_nominales) + len(variables_categoricas_ordinales) + len(variables_continuas) == len(cirrosis.columns)

Qué pasa si de momento sólo nos interesan los signos clínicos.

In [ ]:
cirrosis_signos_clínicos = cirrosis[signos_clinicos]
cirrosis_signos_clínicos.head()

Equivalentemente se podría hacer de la siguiente manera:

In [ ]:
cirrosis[['Ascites', 'Hepatomegaly', 'Spiders', 'Edema']]

De igual manera, podríamos querer las variables de signo clínico y de niveles en sangre.

In [ ]:
cirrosis_variables_médicas = cirrosis[signos_clinicos + nivels_sangre]
cirrosis_variables_médicas.head()

In [ ]:
signos_clinicos + nivels_sangre

Equivalentemente...

In [ ]:
cirrosis[['Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']]

Hack:

In [ ]:
cirrosis[[
    'Ascites', 'Hepatomegaly', 'Spiders', 'Edema', 'Bilirubin',
    'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin'
]]

## Ordenamiento

Tal vez quisieramos ordenar la información de acuerdo a la edad de los pacientes...

In [ ]:
cirrosis_orden_edad_decreciente = cirrosis.sort_values(by='Age', ascending=False)
cirrosis_orden_edad_decreciente

In [ ]:
cirrosis_orden_edad_creciente = cirrosis.sort_values(by='Age', ascending=True)
cirrosis_orden_edad_creciente

Y si me interesa ordenar de acuerdo a varias variables?

Por ejemplo, nos intersa ordenar por etapa y número de días desde diagnóstico a el suceso..

In [ ]:
cirrosis_orden_etapa_creciente_dias_creciente = cirrosis.sort_values(by=['Stage', 'N_Days'])
cirrosis_orden_etapa_creciente_dias_creciente

Como verán no vemos todas las filas por lo que hay que modificar los parámetros de pandas

In [ ]:
pd.set_option('display.max_rows', None)

In [ ]:
cirrosis_orden_etapa_creciente_dias_creciente

¿Qué pasa si quiero el número de días en orden decreciente y la etapa en orden creciente?

In [ ]:
cirrosis_orden_etapa_creciente_dias_decreciente = cirrosis.sort_values(by=['Stage', 'N_Days'], ascending=[True, False])
cirrosis_orden_etapa_creciente_dias_decreciente.head(5)

¿Afecta el orden en el by? Sí, es la jerarquía dijera José...

In [ ]:
cirrosis_orden_dias_decreciente_etapa_creciente = cirrosis.sort_values(by=['N_Days', 'Stage'], ascending=[False, True])
cirrosis_orden_dias_decreciente_etapa_creciente

## Mezclas de dataframes

### Primero voy a generar un par de dataframes para poder ilustrar los distintos tipos de uniones.

In [ ]:
cirrosis.describe()

In [ ]:
cirrosis.describe(
    percentiles=[.1,.2,.25,.5,.75,.9]
)

In [ ]:
cirrosis.describe(
    percentiles=[0,.5,1],
    include='all'
)

In [ ]:
cirrosis.describe(
    percentiles=[0,.5,1],
    include=['object','category']
)

In [ ]:
cirrosis.describe(
    percentiles=[0,.5,1],
    exclude=['number']
)

In [ ]:
cirrosis.isna().sum()

In [ ]:
signos_clinicos = ['Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
nivels_sangre = ['Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
variables_categoricas_nominales = ['Drug', 'Sex', 'Status']
variables_categoricas_ordinales = ['Stage']
variables_continuas = ['N_Days', 'Age']

In [ ]:
df1 = cirrosis.loc[0:418:2, signos_clinicos].copy()
df1

In [ ]:
df2 = cirrosis.loc[2:419:2, signos_clinicos].copy()
df2

In [ ]:
df3 = cirrosis.loc[:,nivels_sangre].copy()
df3

In [ ]:
df4 = cirrosis.loc[1:419:3,variables_categoricas_nominales + variables_categoricas_ordinales]
df4

In [ ]:
df5 = cirrosis.loc[
    [ix for ix in range(1,419) if ix not in range(1,419,3)],
    variables_categoricas_nominales + variables_categoricas_ordinales
]
df5

In [ ]:
df6 = cirrosis.loc[:,variables_continuas]
df6

### Uniones tipo SQL por columnas

Left join: df1 + df5

In [ ]:
pd.merge(
    df1, df5,   # Dataframes izquierdo y derecho
    how='left',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df1','_df5'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Left join: df5 + df1

In [ ]:
pd.merge(
    df5, df1,   # Dataframes izquierdo y derecho
    how='left',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df5','_df1'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Left join: df1 + df2

In [ ]:
pd.merge(
    df1, df2,   # Dataframes izquierdo y derecho
    how='left',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df1','_df2'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Left join: df2 + df1

In [ ]:
pd.merge(
    df2, df1,   # Dataframes izquierdo y derecho
    how='left',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df2','_df1'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Right join: df5 + df1

In [ ]:
pd.merge(
    df5, df1,   # Dataframes izquierdo y derecho
    how='right',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df5','_df1'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Right join: df2 + df1

In [ ]:
pd.merge(
    df2, df1,   # Dataframes izquierdo y derecho
    how='right',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df2','_df1'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Inner join: df4 + df5

In [ ]:
pd.merge(
    df4, df5,   # Dataframes izquierdo y derecho
    how='inner',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df4','_df5'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Inner join: df5 + df4

In [ ]:
pd.merge(
    df5, df4,   # Dataframes izquierdo y derecho
    how='inner',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df5','_df4'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Inner join: df3 + df6

In [ ]:
pd.merge(
    df3, df6,   # Dataframes izquierdo y derecho
    how='inner',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df3','_df6'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Inner join: df2 + df6

In [ ]:
pd.merge(
    df2, df6,   # Dataframes izquierdo y derecho
    how='inner',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df2','_df6'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Outer join: df1 + df2

In [ ]:
pd.merge(
    df1, df2,   # Dataframes izquierdo y derecho
    how='outer',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df1','_df2'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

Outer join: df2 + df1

In [ ]:
pd.merge(
    df2, df1,   # Dataframes izquierdo y derecho
    how='outer',  # left, right, inner, outer, cross
    on='ID',    # Nombre de la columna llave sobre la que se hace el join
    suffixes=('_df2','_df1'),   # Sufijo a agregar a las columna que compartan nombre en los dos dataframes
    indicator=True, # agrega columna _merge
    validate='many_to_one'  # one_to_one, one_to_many, many_to_many
)

### Uniones por índice

Left join: df1 + df2

In [ ]:
df1.join(
    df2,    # Data frame con el cual hacer el join
    how='left', # Tipo de join
    lsuffix='_df1', # Sufijo para columna repetida en data frame de la izquierda
    rsuffix='_df2', # Sufijo para columna repetida en data frame de la derecha
    on='ID' # Sobre donde hacer el join (nombre columna como clave externa)
)

Left join: df2 + df1

In [ ]:
df2.join(
    df1,    # Data frame con el cual hacer el join
    how='left', # Tipo de join
    lsuffix='_df1', # Sufijo para columna repetida en data frame de la izquierda
    rsuffix='_df2', # Sufijo para columna repetida en data frame de la derecha
    on='ID' # Sobre donde hacer el join (nombre columna como clave externa)
)

Right join: df2 + df1

In [ ]:
df2.join(
    df1,    # Data frame con el cual hacer el join
    how='right', # Tipo de join
    lsuffix='_df2', # Sufijo para columna repetida en data frame de la izquierda
    rsuffix='_df1', # Sufijo para columna repetida en data frame de la derecha
    on='ID' # Sobre donde hacer el join (nombre columna como clave externa)
)

In [ ]:
df2.join(
    df1,    # Data frame con el cual hacer el join
    how='right', # Tipo de join
    lsuffix='_df2', # Sufijo para columna repetida en data frame de la izquierda
    rsuffix='_df1', # Sufijo para columna repetida en data frame de la derecha
)

In [ ]:
df2

on='ID'
df2.set_index('ID')

### Concatenar/pegar

Apilar filas: df1 + df2

In [ ]:
pd.concat(
    [df1, df2], # Lista de dataframes a pegar
    axis=0,    # 0 para filas, 1 para columnas
    ignore_index=True,  # Conserva los indices de los dataframes o reindexa de 0 a el número de filas
    sort=False, # Ordenar por índice después de unir
)

In [ ]:
pd.concat(
    [df1, df2], # Lista de dataframes a pegar
    axis=0,    # 0 para filas, 1 para columnas
    ignore_index=False,  # Conserva los indices de los dataframes o reindexa de 0 a el número de filas
    sort=False, # Ordenar por índice después de unir
)

In [ ]:
pd.concat(
    [df1, df2], # Lista de dataframes a pegar
    axis=0,    # 0 para filas, 1 para columnas
    ignore_index=True,  # Conserva los indices de los dataframes o reindexa de 0 a el número de filas
    sort=True, # Ordenar por índice después de unir
)

In [ ]:
pd.concat(
    [df1, df2], # Lista de dataframes a pegar
    axis=0,    # 0 para filas, 1 para columnas
    ignore_index=False,  # Conserva los indices de los dataframes o reindexa de 0 a el número de filas
    sort=True, # Ordenar por índice después de unir
)

In [ ]:
pd.concat(
    [df1, df2], # Lista de dataframes a pegar
    axis=0,    # 0 para filas, 1 para columnas
    ignore_index=True,  # Conserva los indices de los dataframes o reindexa de 0 a el número de filas
    sort=True, # Ordenar por índice después de unir
)

Pegar columnas: df3 + df6

In [ ]:
pd.concat(
    [df3, df6],
    axis=1
)

Pegar columnas: df1 + df4

In [ ]:
pd.concat(
    [df1, df4],
    axis=1
)

¿Si queremos recuperar todo el dataframe?

* df1 y df2

In [ ]:
pd.concat(
    [df1, df2],
    axis=0
).sort_index()

* df3

df3

* df4 y df5

In [ ]:
pd.concat(
    [df4, df5],
    axis=0
).sort_index()

* df6

In [ ]:
df6

* Todos juntos

In [ ]:
pd.concat(
    [
        pd.concat(
            [df1, df2],
            axis=0
        ).sort_index(),
        df3,
        pd.concat(
            [df4, df5],
            axis=0
        ).sort_index(),
        df6
    ],
    axis=1
)[cirrosis.columns]

## Agregación de datos

Quitamos la variable 'ID' de índice y la pasamos a columna para que funciones está sección

In [ ]:
cirrosis.reset_index(inplace=True)

### Named aggregation

Primero nos aseguramos que algunas columnas sean numéricas

In [ ]:
num_cols = ['Bilirubin', 'Albumin', 'Cholesterol', 'N_Days', 'Age']
cirrosis[num_cols] = cirrosis[num_cols].apply(pd.to_numeric, errors="coerce")

Agrupamos por la variable 'Status' y calculamos variables de interés

In [ ]:
res_status = (
    cirrosis
    .groupby(
        # Etiquetas de agrupación, puede ser str, list o dict
        by='Status',
        # Si las claves quedan como índice (tidyverse)
        as_index=True,
        # Incluir grupos donde 'Status' es NaN
        dropna=False,
        # Ordenar las claves de grupo
        sort=True,
        # Agrupar por categorías o sólo observadas
        observed=False
    )
    .agg(
        # Conteo de filas con ID no nulo
        n_pacientes=('ID', 'count'),
        # Mediana de la bilirubina
        bili_mediana=('Bilirubin', 'median'),
        # Cuantil .95 usando función lambda
        bili_p95=('Bilirubin', lambda s: s.quantile(0.95)),
        # Media aritmética de la albumina
        alb_media=('Albumin', 'mean'),
        # Mediana del número de días de seguimiento
        dias_medianos=('N_Days', 'median')
    )
    .reset_index()
)

Primeras filas de las estadísticas calculadas

In [ ]:
print(res_status.head())

### Named aggregation con diccionario

In [ ]:
res_drug_edema = (
    cirrosis
    .groupby(['Drug','Edema'], as_index=False, dropna=False)
    .agg({
        # Múltiples funciones en una misma columna
        'Bilirubin': ['median', 'mean', 'std', 'max'],
        'Albumin': ['median', 'mean'],
        # Para contar filas se debe usar 'size'
        'ID': 'count'
    })
)

Aplanamos el MultiIndex de columnas, esto es útil para guardar la base o generar reportes.

In [ ]:
res_drug_edema.columns = [
    # Renombrar las columnas para que tengan etiquetas del tipo variable_estadística
    '_'.join([c for c in map(str, col) if c and c != ''])
    for col in res_drug_edema.columns.values
]

Dataframe resultante

In [ ]:
print(res_drug_edema.head())

### Tabla dinámica

El objetivo es tener tabla por filas **Status** y columnas **Edema** con la **mediana de la bilirubina**, totales y ceros donde no hay observaciones.

In [ ]:
tabla_bili = pd.pivot_table(
    # Dataframe de entrada
    data=cirrosis,
    # Variable utilizada para las filas
    index='Status',
    # Variable utilizada para las columnas
    columns='Edema',
    # Variable sobre la que queremos obtener datos agregados
    values='Bilirubin',
    # Función de agregación
    aggfunc='median',
    # Añadir totales
    margins=True,
    # Nombre de los totales
    margins_name='Total',
    # Imputación de valores no numéricos para la presentación
    fill_value=0,
    # Conserva combinaciones con NA
    dropna=False,
    # Ver nota de 'observed'
    observed=False
)
print(tabla_bili)

Veamos que representan los totaltes en la tabla de arriba:

In [ ]:
cirrosis[cirrosis['Status']=='D']['Bilirubin'].median()

In [ ]:
cirrosis[cirrosis['Edema']=='N']['Bilirubin'].median()

* pivot_table es ideal para matrices con una o varias medidas ya que incluye fácilmente los totales
* groupby es más general para pipelines y salidas 'largas'

### Tablas de contingencia

Conteo bruto

In [ ]:
xt = pd.crosstab(
    # Filas
    index=cirrosis['Status'],
    # Columnas
    columns=cirrosis['Drug'],
    # Si None se hace conteo, si pasa una serie se debe indicar 'aggfunc'
    values=None,
    # Función de agregación cuando 'values' no es None
    aggfunc=None,
    # Proporciones, puede ser True | False | 'all' | 'index' | 'columns'
    normalize=False,
    # Totales
    margins=True,
    # Nombre de totales
    margins_name='Total',
    # Conserva categorías NA
    dropna=False
)
print(xt)

Proporciones por fila

In [ ]:
xt_rowprop = pd.crosstab(
    cirrosis['Status'],
    cirrosis['Drug'],
    normalize='index',
    margins=True,
    margins_name='Total'
)
print(xt_rowprop)

Proporciones por columna

In [ ]:
xt_colprop = pd.crosstab(
    cirrosis['Status'],
    cirrosis['Drug'],
    normalize='columns',
    margins=True,
    margins_name='Total'
)
print(xt_colprop)

### Transformaciones

Se desea centrar la **bilirubina** por grupo **Status**. Primero calculamos la media por status:

In [ ]:
bili_media_por_status = cirrosis.groupby('Status')['Bilirubin'].transform('mean')

El método transform calcula la media por cada grupo y la 'expande' fila a fila. Ahora podemos centrar la bilirubina por grupo:

In [ ]:
cirrosis['Bilirubin_centered_status'] = cirrosis['Bilirubin'] - bili_media_por_status

In [ ]:
cirrosis[['Bilirubin', 'Bilirubin_centered_status']].mean()

### Agregaciones ponderadas

En esta ocación queremos obtener una media ponderada de la **bilirubina** por **Status** utilizando **N_Days** como peso. Para ello primero definimos una función auxiliar:

In [ ]:
def weighted_mean(x, w):
    # x: serie de valores;
    # w: serie de pesos (alineada por índice)
    return np.average(x, weights=w)

Ahora procedemos a realizar la agregación:

In [ ]:
res_wavg = (
    cirrosis
    .dropna(subset=['Bilirubin','N_Days'])
    .groupby('Status')
    .apply(lambda g: weighted_mean(g['Bilirubin'], g['N_Days']), include_groups=False)    # 'g' es sub-DF del grupo
    .reset_index(name='bili_media_ponderada')
)
print(res_wavg)

### Binning

Ahora queremos agrupar pacientes en ventanas de 100 días de seguimiento y resumir la bilirubina. Primero construímos los bins de 0 al máximo de días con paso 'step'.

In [ ]:
step = 100
max_nd = int(np.nanmax(cirrosis['N_Days'])) if cirrosis['N_Days'].notna().any() else 0
bins = np.arange(0, max_nd + step, step)

In [ ]:
bins

Ahora etiquetamos cada fila en su bin:

In [ ]:
cirrosis['N_Days_bin'] = pd.cut(
    # Variable continua
    cirrosis['N_Days'],
    # Bordes de intervalos
    bins=bins,
    # Incluir el extremo inferior en el primer bin
    include_lowest=True,
    # Intervalos cerrados a la izquierda [a, b)
    right=False
)

Realizamos la agregación por bin y la clave Status:

In [ ]:
res_bins = (
    cirrosis
    .groupby(['N_Days_bin', 'Status'], dropna=False, observed=True)
    .agg(
        n=('ID', 'count'),
        bili_med=('Bilirubin', 'median'),
        alb_med=('Albumin', 'median')
    )
    .reset_index()
    .sort_values(['N_Days_bin', 'Status'])
)
print(res_bins.head(10))

Los parámetros utilizados en pd.cut son:
* **bins**: int (número de cortes) o secuencia de bordes
* **labels**: etiquetas personalizadas para cada intervalo
* **right**: True para intervalos del tipo (a, b] y False para intervalos del tipo [a, b)
* **include_lowest**: incluye el límite inferior en el primer intervalo

### Agregación multinivel

Queremos ordenar combinaciones de **Drug x Edema** por **mediana de bilirubina** en orden decreciente.

In [ ]:
rank_bili = (
    cirrosis
    .groupby(['Drug', 'Edema'], dropna=False, as_index=False)
    .agg(bili_med=('Bilirubin', 'median'), n=('ID', 'count'))
    .sort_values(
        by=['bili_med','n'],
        ascending=[False, False],
        kind='mergesort'
    )
    .reset_index(drop=True)
)

print(rank_bili.head(10))

El parámetro **kind** hace referencia al método de ordenamiento. Puede ser quicksort, mergesort o heapsort. En el ejemplo utilizamos mergesort porque es **estable**

### Agregación con Grouper temporal

El dataset tiene **N_days** pero no es una fecha calendario. Supongamos para fines didácticos que tenemos una columna real de fechas, por ejemplo **visit_date**, podríamos agrupar por frecuencia de calendario:

In [ ]:
# Ejemplo ilustrativo si existiera 'visit_date' en datetime64[ns]
cirrosis['visit_date'] = pd.to_datetime(cirrosis['visit_date'])

res_time = (
    cirrosis
    .groupby(pd.Grouper(key='visit_date', freq='M'))  # 'M' mensual, también 'W','Q','Y','D'
    .agg(n=('ID', 'count'), bili_med=('Bilirubin', 'median'))
    .reset_index()
)

### Acordeon

In [ ]:
acordeon = {
    # Conteos
    'conteos_por_status': cirrosis.groupby('Status', dropna=False).size().reset_index(name='n'),

    # Estadísticos por tratamiento
    'stats_drug': cirrosis.groupby('Drug', dropna=False).agg(
        n=('ID', 'count'),
        bili_med=('Bilirubin', 'median'),
        alb_med=('Albumin', 'median'),
        chol_med=('Cholesterol', 'median')
    )
    .reset_index(),

    # Tabla dinámica: mediana de Bilirrubina por Status (filas) y Edema (columnas)
    'pivot_bili': pd.pivot_table(
        cirrosis,
        index='Status',
        columns='Edema',
        values='Bilirubin',
        aggfunc='median',
        margins=True,
        margins_name='Total',
        fill_value=0
    ),

    # Crosstab proporciones por fila (Status x Drug)
    'xt_rowprop': pd.crosstab(
        cirrosis['Status'],
        cirrosis['Drug'],
        normalize='index',
        margins=True,
        margins_name='Total'
    )
}

for nombre, tabla in acordeon.items():
    print(f"\n=== {nombre} ===")
    print(tabla.head())